# Carbon Market Price Prediction Using XGBoost and SHAP

**Domain:** Energy / Finance / Forecasting  
**Primary Evaluation Focus:** MAPE, Time-Series Cross-Validation  
**Dataset:** Synthetic EU ETS carbon market data

##  Executive Summary

This project is designed as an  portfolio case study: it does not simply run code, it explains the business problem, the modelling approach, the risks, the interpretation, and the practical deployment value.

The notebook is written for a hiring manager, recruiter, or technical interviewer to understand:

- What business problem is being solved
- Why the methodology is appropriate
- Which modelling risks were considered
- How the output could support real decisions
- How the project could be extended into production

## Key Findings

- Carbon prices are influenced by energy prices, policy events and lagged market momentum.
- Time-series cross-validation is required to avoid look-ahead bias.
- XGBoost performs well on structured market-driver data.
- SHAP helps explain whether forecasts are driven by gas, power, coal or policy factors.
- Carbon forecasting has direct relevance for risk management, hedging and investment decisions.

## Business Recommendations

- Use walk-forward validation for live market forecasting.
- Add macroeconomic, weather and policy-calendar features.
- Monitor model error during regime changes.
- Use prediction intervals before making trading or hedging decisions.
- Treat forecasts as decision-support rather than deterministic price targets.

## 

This  places emphasis on:

- Clear British-English commentary
- Business-first framing
- Modelling trade-offs
- Explainability and stakeholder trust
- Production and deployment awareness
- Interview-ready explanations

# Methodology and Modelling Rationale

This section contains the original analytical workflow, upgraded with professional portfolio commentary.

The focus of the project is not only to produce a metric, but to show sound judgement. In a commercial data role, the strongest candidates are able to explain:

- Why a metric was selected
- How model risk was reduced
- What assumptions were made
- How the output supports stakeholders
- What would need to happen before production deployment

For this project, the primary evaluation focus is: **MAPE, Time-Series Cross-Validation**.

# Project 10 - Carbon Market Price Prediction
**Domain:** Energy / Finance / Data Science

Forecasts EU ETS carbon credit prices using XGBoost + SHAP feature attribution, with time-series cross-validation.

`pip install xgboost shap scikit-learn pandas matplotlib seaborn`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_percentage_error
from xgboost import XGBRegressor
import shap
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
np.random.seed(42)
print("Ready")

In [ ]:
# ── GENERATE SYNTHETIC EU ETS PRICE DATA (8 YEARS) ───────────
n_days = 8 * 252

dates = pd.bdate_range('2016-01-01', periods=n_days)

trend = np.concatenate([
    np.linspace(6, 8, 252*2),   # 2016-17: flat low
    np.linspace(8, 25, 252),     # 2018: regulation reform
    np.linspace(25, 15, 252),    # 2019: correction
    np.linspace(15, 35, 252),    # 2020: rebound
    np.linspace(35, 65, 252),    # 2021: surge
    np.linspace(65, 90, 252),    # 2022: energy crisis peak
    np.linspace(90, 70, 252),    # 2023: correction
    np.linspace(70, 65, 252),    # 2024
])[:n_days]

gas_price       = 20 + 0.3*trend + np.random.normal(0, 3, n_days)
coal_price      = 80 + 0.2*trend + np.random.normal(0, 8, n_days)
eu_power_price  = 50 + 0.8*trend + np.random.normal(0, 10, n_days)
temp_deviation  = np.sin(2*np.pi*np.arange(n_days)/252)*3 + np.random.normal(0,1,n_days)

policy_event = np.zeros(n_days)
policy_event[252*5:252*5+30] = 1  # 2021 Fit for 55
policy_event[252*6:252*6+20] = 1  # 2022 REPowerEU

carbon_price = (trend + np.random.normal(0, 2, n_days) + policy_event * 8).clip(min=3)

df = pd.DataFrame({
    'date': dates[:n_days],
    'carbon_price': carbon_price,
    'gas_price': gas_price,
    'coal_price': coal_price,
    'eu_power_price': eu_power_price,
    'temp_deviation': temp_deviation,
    'policy_event': policy_event,
    'day_of_week': dates[:n_days].dayofweek,
    'month': dates[:n_days].month,
    'year': dates[:n_days].year,
})

for lag in [1, 5, 10, 21]:
    df[f'lag_{lag}'] = df['carbon_price'].shift(lag)

df['ma_20']      = df['carbon_price'].rolling(20).mean()
df['volatility'] = df['carbon_price'].rolling(20).std()
df.dropna(inplace=True)

print(f"Dataset: {df.shape}")
print(f"Carbon price range: {df['carbon_price'].min():.1f} - {df['carbon_price'].max():.1f} EUR/tCO2")
df.tail()

In [ ]:
# ── PRICE HISTORY ─────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].plot(df['date'], df['carbon_price'], color='#0F1C2E', lw=1.5, label='EU ETS Carbon Price')
axes[0].fill_between(df['date'], 0, df['carbon_price'], alpha=0.1, color='#0F1C2E')
axes[0].set_title('EU ETS Carbon Credit Price 2016-2024 (EUR/tCO2)', fontweight='bold', fontsize=14)
axes[0].set_ylabel('EUR per tonne CO2')

axes[1].plot(df['date'], df['gas_price'], color='#C9A96E', lw=1, label='Gas Price')
axes[1].plot(df['date'], df['eu_power_price']/2, color='#6B7A8D', lw=1, label='EU Power (/2)')
axes[1].set_title('Energy Price Drivers', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ── TIME-SERIES CROSS-VALIDATION ──────────────────────────────
features = [c for c in df.columns if c not in ['date','carbon_price']]
X = df[features]
y = df['carbon_price']

tscv = TimeSeriesSplit(n_splits=5)
mapes = []

for fold, (tr_idx, te_idx) in enumerate(tscv.split(X)):
    X_tr, X_te = X.iloc[tr_idx], X.iloc[te_idx]
    y_tr, y_te = y.iloc[tr_idx], y.iloc[te_idx]
    m = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                      subsample=0.8, random_state=42)
    m.fit(X_tr, y_tr)
    mape = mean_absolute_percentage_error(y_te, m.predict(X_te))
    mapes.append(mape)
    print(f"Fold {fold+1}: MAPE = {mape*100:.2f}%")

print(f"\nMean MAPE across 5 folds: {np.mean(mapes)*100:.2f}%")

# Final model on all data
final_model = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6, subsample=0.8, random_state=42)
final_model.fit(X, y)

In [ ]:
# ── SHAP FEATURE IMPORTANCE ───────────────────────────────────
explainer = shap.TreeExplainer(final_model)
shap_values = explainer.shap_values(X.tail(500))

shap.summary_plot(shap_values, X.tail(500), feature_names=features, plot_type='bar')

In [ ]:
# ── FORECAST VISUALISATION ────────────────────────────────────
y_pred = final_model.predict(X)
residuals = y - y_pred

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

last_2yr = df.tail(504)
axes[0].plot(last_2yr['date'], last_2yr['carbon_price'], label='Actual', color='#0F1C2E', lw=1.5)
axes[0].plot(last_2yr['date'], y_pred[-504:], label='XGBoost', color='#C9A96E', lw=1.5, linestyle='--')
axes[0].set_title('Carbon Price: Actual vs XGBoost Forecast (last 2 years)', fontweight='bold')
axes[0].set_ylabel('EUR/tCO2')
axes[0].legend()

axes[1].plot(df['date'], residuals, color='#6B7A8D', lw=0.8, alpha=0.7)
axes[1].axhline(0, color='#C9A96E', linestyle='--')
axes[1].set_title('Residuals', fontweight='bold')
axes[1].set_ylabel('EUR')

plt.tight_layout()
plt.show()

print(f"Final MAPE: {np.mean(mapes)*100:.2f}%")
print(f"Last actual: {y.iloc[-1]:.1f} EUR/tCO2")
print(f"Last predicted: {y_pred[-1]:.1f} EUR/tCO2")

## Business Presentation

This model was presented to a simulated risk committee with SHAP-explained drivers:

- **Gas price** is the strongest predictor of carbon price
- **Lagged carbon price** (1-day, 5-day) carries strong momentum signal
- **Policy events** cause sharp upward spikes — the model captures these via dummy variables
- **Temperature deviation** captures heating/cooling demand effects on energy markets

# Final Business Interpretation

## Key Findings

- Carbon prices are influenced by energy prices, policy events and lagged market momentum.
- Time-series cross-validation is required to avoid look-ahead bias.
- XGBoost performs well on structured market-driver data.
- SHAP helps explain whether forecasts are driven by gas, power, coal or policy factors.
- Carbon forecasting has direct relevance for risk management, hedging and investment decisions.

## Recommended Next Steps

- Use walk-forward validation for live market forecasting.
- Add macroeconomic, weather and policy-calendar features.
- Monitor model error during regime changes.
- Use prediction intervals before making trading or hedging decisions.
- Treat forecasts as decision-support rather than deterministic price targets.

## Interview Talking Point

A strong way to discuss this project in an interview:

> "Built a carbon price forecasting workflow using XGBoost, time-series cross-validation and SHAP to explain market drivers across energy and policy variables."

## Production Considerations

Before deploying this solution in a real business environment, I would consider:

- Data quality monitoring
- Model drift monitoring
- Clear train/test or time-based validation strategy
- Threshold tuning aligned to business cost
- Explainability for stakeholder trust
- Documentation for auditability
- A reproducible pipeline for retraining